<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER_fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from collections import Counter
from pathlib import Path
import pandas as pd
import subprocess
import tempfile
import shutil
import json
import sys
import os
import re

IS_COLAB_VM = (
    "google.colab" in sys.modules
    and os.path.exists("/content")
    and os.environ.get("COLAB_RELEASE_TAG") is not None
)

if IS_COLAB_VM:
    from google.colab import userdata
else:
    userdata = None

### Clone Corpus2GeoRAG Repository

In [13]:
IS_LOCAL = os.name == "nt"

LOCAL_REPO_PATH = Path.home() / "Corpus2GeoRAG"
repo_dir = (
    LOCAL_REPO_PATH
    if IS_LOCAL
    else Path("/content/Corpus2GeoRAG")
)

REPO_PATH = str(repo_dir)

if not repo_dir.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/EtzionR/Corpus2GeoRAG.git",
            REPO_PATH
        ],
        check=True
    )

elif not (repo_dir / ".git").exists():
    raise RuntimeError(
        f"{REPO_PATH} exists but is not a Git checkout."
    )

else:
    # Download remote commit information.
    subprocess.run(
        ["git", "fetch", "origin"],
        cwd=repo_dir,
        check=True
    )

    # Update only if Git can fast-forward safely.
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=repo_dir,
        check=True
    )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Working folder:", REPO_PATH)

Runtime: Local Windows
Working folder: C:\Users\idosc\Corpus2GeoRAG


### Cohere cleaning

this step is to clear Cohere extracted entitiies from refrences.

#### Cohere GT upload and cleaning

Uploading and preparing GT

In [14]:
# ============================================================
# 1. FILE PATHS
# ============================================================

repo_dir = Path(REPO_PATH)

evaluation_dir = (
    repo_dir
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

ground_truth_dir = evaluation_dir / "ground_truth"

cohere_path = ground_truth_dir / "cohere_entities.parquet"
subset_path = evaluation_dir / "evaluation_subset.parquet"
corpus_path = repo_dir / "CollectCorpus" / "DATA.json"

required_files = {
    "Cohere ground truth": cohere_path,
    "Evaluation subset": subset_path,
    "Wikipedia corpus": corpus_path
}

for name, path in required_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

# ============================================================
# 2. LOAD ORIGINAL DATA
# ============================================================

cohere_gt_original = pd.read_parquet(cohere_path)
evaluation_df_original = pd.read_parquet(subset_path)

required_entity_columns = {
    "Source", "Index", "Text",
    "word", "entity_group", "start", "end"
}

required_subset_columns = {"Source", "Index", "Text"}

for name, df, required in [
    ("Cohere GT", cohere_gt_original, required_entity_columns),
    ("Evaluation subset", evaluation_df_original, required_subset_columns)
]:
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"{name} missing columns: {missing}")

    if df[list(required_subset_columns)].isna().any().any():
        raise ValueError(f"{name} contains missing identifiers.")

if evaluation_df_original.empty:
    raise ValueError("Evaluation subset is empty.")

if evaluation_df_original.duplicated(["Source", "Index"]).any():
    raise ValueError("Duplicate Source/Index pairs.")

# ============================================================
# 3. LOAD WIKIPEDIA CORPUS
# ============================================================

with open(corpus_path, "r", encoding="utf-8") as f:
    original_corpus = json.load(f)

if not isinstance(original_corpus, dict):
    raise ValueError("Expected a dictionary of Wikipedia pages.")

required_pages = set(evaluation_df_original["Source"])
missing_pages = required_pages - set(original_corpus)

# ============================================================
# 4. REPRODUCE THE ORIGINAL COHERE PREPROCESSING
# ============================================================

TITLE = "=="
MIN_LENGTH = 5

def reconstruct_paragraphs(page_text):
    """
    EXACT preprocessing used before generating
    the original Cohere ground truth.

    Do not remove References or External links here.
    Their removal belongs to the next Cleaning cell.
    """

    paragraphs = page_text.split("\n")

    return [
        text
        for text in paragraphs
        if not text.startswith(TITLE) and len(text) >= MIN_LENGTH
    ]


reconstructed_paragraphs = {}

for page in required_pages - missing_pages:

    page_text = original_corpus[page]

    if not isinstance(page_text, str):
        raise ValueError(
            f"Unexpected article content type: {page}"
        )

    reconstructed_paragraphs[page] = reconstruct_paragraphs(
        page_text
    )

# ============================================================
# 5. VERIFY ORIGINAL PARAGRAPH INDEX ALIGNMENT
# ============================================================

misaligned_pages = set()
alignment_examples = []

for row in evaluation_df_original[
    ["Source", "Index", "Text"]
].itertuples(index=False):

    page = row.Source

    if page in missing_pages:
        continue

    index = int(row.Index)
    paragraphs = reconstructed_paragraphs[page]

    if (
        index < 0
        or index >= len(paragraphs)
        or paragraphs[index] != row.Text
    ):
        misaligned_pages.add(page)

        if len(alignment_examples) < 10:
            alignment_examples.append({
                "Source": page,
                "Index": index,
                "Expected": row.Text[:150],
                "Reconstructed": (
                    paragraphs[index][:150]
                    if 0 <= index < len(paragraphs)
                    else "Index out of range"
                )
            })

# ============================================================
# 6. VERIFY COHERE AGAINST THE EVALUATION SUBSET
# ============================================================

cohere_paragraphs_original = cohere_gt_original[
    ["Source", "Index", "Text"]
].drop_duplicates()

alignment_check = cohere_paragraphs_original.merge(
    evaluation_df_original[["Source", "Index", "Text"]],
    on=["Source", "Index", "Text"],
    how="left",
    indicator=True
)

cohere_conflict_pages = set(
    alignment_check.loc[
        alignment_check["_merge"] != "both",
        "Source"
    ]
)

# ============================================================
# 7. EXCLUDE ONLY GENUINE CONFLICTS
# ============================================================

excluded_pages = (
    missing_pages
    | misaligned_pages
    | cohere_conflict_pages
)

valid_pages = required_pages - excluded_pages

if not valid_pages:
    raise ValueError("No valid Wikipedia articles remain.")

evaluation_df = (
    evaluation_df_original[
        evaluation_df_original["Source"].isin(valid_pages)
    ]
    .copy()
    .reset_index(drop=True)
)

cohere_gt = (
    cohere_gt_original[
        cohere_gt_original["Source"].isin(valid_pages)
    ]
    .copy()
    .reset_index(drop=True)
)

cohere_paragraphs = cohere_gt[
    ["Source", "Index", "Text"]
].drop_duplicates()

relevant_pages = {
    page: original_corpus[page]
    for page in sorted(valid_pages)
}

original_paragraphs = {
    page: reconstructed_paragraphs[page]
    for page in valid_pages
}

del original_corpus

# ============================================================
# 8. FINAL ALIGNMENT VERIFICATION
# ============================================================

evaluation_keys = set(
    zip(
        evaluation_df["Source"],
        evaluation_df["Index"],
        evaluation_df["Text"]
    )
)

cohere_keys = set(
    zip(
        cohere_paragraphs["Source"],
        cohere_paragraphs["Index"],
        cohere_paragraphs["Text"]
    )
)

if not cohere_keys.issubset(evaluation_keys):
    raise ValueError("Cohere paragraph alignment failed.")

for row in evaluation_df.itertuples(index=False):

    reconstructed = original_paragraphs[row.Source]

    if reconstructed[int(row.Index)] != row.Text:
        raise ValueError(
            f"Final alignment failed: "
            f"{row.Source}, index {row.Index}"
        )

# ============================================================
# 9. SUMMARY
# ============================================================

print("===== COHERE GT PREPARATION =====")

print(f"Original evaluation pages: {len(required_pages):,}")
print(f"Original evaluation paragraphs: {len(evaluation_df_original):,}")
print(f"Original Cohere entities: {len(cohere_gt_original):,}")

print("\n===== CORPUS ALIGNMENT =====")

print(f"Missing pages: {len(missing_pages):,}")
print(f"Modified/misaligned pages: {len(misaligned_pages):,}")
print(f"Cohere/evaluation conflicts: {len(cohere_conflict_pages):,}")
print(f"Total excluded pages: {len(excluded_pages):,}")

print("\n===== PREPARED DATASETS =====")

print(f"Retained Wikipedia pages: {len(valid_pages):,}")
print(f"Retained evaluation paragraphs: {len(evaluation_df):,}")
print(f"Retained Cohere entities: {len(cohere_gt):,}")
print(f"Excluded Cohere entities: {len(cohere_gt_original) - len(cohere_gt):,}")

if excluded_pages:
    print("\n===== EXCLUDED ARTICLES =====")

    for page in sorted(excluded_pages)[:20]:
        reasons = []

        if page in missing_pages:
            reasons.append("Missing from corpus")

        if page in misaligned_pages:
            reasons.append("Paragraph content/index mismatch")

        if page in cohere_conflict_pages:
            reasons.append("Cohere/evaluation mismatch")

        print(f"{page}: {', '.join(reasons)}")

if alignment_examples:
    print("\n===== ALIGNMENT MISMATCH EXAMPLES =====")
    display(pd.DataFrame(alignment_examples))

print("\nPreparation completed.")
print("Original datasets remain unchanged.")

display(cohere_gt.head(10))

===== COHERE GT PREPARATION =====
Original evaluation pages: 191
Original evaluation paragraphs: 6,565
Original Cohere entities: 7,326

===== CORPUS ALIGNMENT =====
Missing pages: 30
Modified/misaligned pages: 0
Cohere/evaluation conflicts: 0
Total excluded pages: 30

===== PREPARED DATASETS =====
Retained Wikipedia pages: 161
Retained evaluation paragraphs: 5,416
Retained Cohere entities: 6,799
Excluded Cohere entities: 527

===== EXCLUDED ARTICLES =====
Anti-Russian sentiment: Missing from corpus
Bridge over Saint Volodymyr Descent: Missing from corpus
Chernobyl exclusion zone: Missing from corpus
Cyber Partisans: Missing from corpus
Davyd Zhvania: Missing from corpus
Freedom on Fire: Ukraine's Fight for Freedom: Missing from corpus
Fyodor Berezin: Missing from corpus
Great Construction: Missing from corpus
Intermarium: Missing from corpus
Ivano-Frankivsk Oblast: Missing from corpus
Kamianka Sich: Missing from corpus
Kharkiv State Academy of Culture: Missing from corpus
Mejlis of the

,entity_group,score,word,start,end,Source,Index,Text
0,ORG,NaN,Ternopil Battalion,4,22,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
1,ORG,NaN,Special Tasks Patrol Police,51,78,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
2,ORG,NaN,Ministry of Internal Affairs of Ukraine,95,134,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
3,LOC,NaN,Ternopil,162,170,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
4,MISC,NaN,War in Donbass,218,232,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
5,LOC,NaN,Ternopil Oblast,100,115,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
6,MISC,NaN,Euromaidan,216,226,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
7,PER,NaN,Vyacheslav Kizilov,296,314,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
8,MISC,NaN,ATO,420,423,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
9,MISC,NaN,ATO,491,494,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."


Cleaning

In [15]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

TITLE = "=="
MIN_LENGTH = 5

STOP_HEADINGS = {"references", "external links"}

heading_pattern = re.compile(
    r"(?P<marks>={2,})\s*(?P<heading>.*?)\s*(?P=marks)"
)

output_path = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
    / "ground_truth"
    / "cohere_entities_cleaned.parquet"
)

# ============================================================
# 2. FIND CUTOFFS USING ORIGINAL PARAGRAPH INDICES
# ============================================================

cutoff_positions = {}
cutoff_headings = {}
cutoff_counts = Counter()

for page, page_text in relevant_pages.items():

    original_index = 0
    cutoff = len(original_paragraphs[page])
    cutoff_heading = None

    for text in page_text.split("\n"):

        heading_match = heading_pattern.fullmatch(text.strip())

        if heading_match:
            heading_text = heading_match.group("heading").strip()

            normalized_heading = " ".join(
                heading_text.casefold().split()
            )

            if normalized_heading in STOP_HEADINGS:
                cutoff = original_index
                cutoff_heading = normalized_heading
                cutoff_counts[normalized_heading] += 1
                break

        # EXACT original Cohere preprocessing.
        # Count only lines that originally received an Index.
        if not text.startswith(TITLE) and len(text) >= MIN_LENGTH:
            original_index += 1

    cutoff_positions[page] = cutoff
    cutoff_headings[page] = cutoff_heading

# ============================================================
# 3. IDENTIFY PARAGRAPHS AFTER THE CUTOFF
# ============================================================

excluded_paragraphs = set()

for row in evaluation_df[
    ["Source", "Index", "Text"]
].itertuples(index=False):

    page = row.Source
    index = int(row.Index)

    # Alignment must still match the original Cohere data.
    if original_paragraphs[page][index] != row.Text:
        raise ValueError(
            f"Alignment failed: {page}, index {index}"
        )

    if index >= cutoff_positions[page]:
        excluded_paragraphs.add((page, index))

# ============================================================
# 4. REMOVE ONLY ENTITIES FROM EXCLUDED PARAGRAPHS
# ============================================================

remove_mask = pd.Series(
    [
        (source, int(index)) in excluded_paragraphs
        for source, index in zip(
            cohere_gt["Source"],
            cohere_gt["Index"]
        )
    ],
    index=cohere_gt.index,
    dtype=bool
)

removed_entities = cohere_gt.loc[remove_mask].copy()

cohere_gt_cleaned = (
    cohere_gt.loc[~remove_mask]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# 5. VERIFY THE RESULTS
# ============================================================

original_count = len(cohere_gt)
removed_count = len(removed_entities)
cleaned_count = len(cohere_gt_cleaned)

if original_count != removed_count + cleaned_count:
    raise ValueError("Entity counts do not match.")

for row in cohere_gt_cleaned[
    ["Source", "Index", "Text", "word", "start", "end"]
].itertuples(index=False):

    if (row.Source, int(row.Index)) in excluded_paragraphs:
        raise ValueError("An excluded paragraph remains.")

    start = int(row.start)
    end = int(row.end)

    if row.Text[start:end] != row.word:
        raise ValueError(
            f"Entity offset mismatch: "
            f"{row.Source}, index {row.Index}"
        )

# ============================================================
# 6. SAVE SEPARATE CLEANED GT
# ============================================================

output_path.parent.mkdir(parents=True, exist_ok=True)

save_required = True

if output_path.exists():
    existing = pd.read_parquet(output_path)

    if existing.equals(cohere_gt_cleaned):
        save_required = False
        print("Existing cleaned GT already matches.")

if save_required:
    temporary_path = output_path.with_suffix(".tmp")

    cohere_gt_cleaned.to_parquet(
        temporary_path,
        engine="pyarrow",
        compression="zstd",
        index=False
    )

    os.replace(temporary_path, output_path)

    print("Cleaned Cohere GT saved successfully.")

# ============================================================
# 7. CLEANING SUMMARY
# ============================================================

removed_percentage = (
    removed_count / original_count * 100
    if original_count else 0
)

print("\n===== COHERE CLEANING SUMMARY =====")

print(f"Original entities in valid articles: {original_count:,}")
print(f"Removed reference entities: {removed_count:,}")
print(f"Cleaned entities: {cleaned_count:,}")
print(f"Entities removed: {removed_percentage:.2f}%")

print(f"Evaluation paragraphs excluded: {len(excluded_paragraphs):,}")
print(f"Pages with References cutoff: {cutoff_counts['references']:,}")
print(f"Pages with External links cutoff: {cutoff_counts['external links']:,}")

print("\n===== REMOVED ENTITIES BY CUTOFF =====")

if not removed_entities.empty:

    removed_by_heading = (
        removed_entities["Source"]
        .map(cutoff_headings)
        .value_counts()
    )

    for heading, count in removed_by_heading.items():
        print(f"{heading}: {count:,} entities")

    print("\n===== REMOVED ENTITY PREVIEW =====")

    display(
        removed_entities[
            ["Source", "Index", "word", "entity_group"]
        ].head(20)
    )

print("\n===== CLEANED GT PREVIEW =====")
display(cohere_gt_cleaned.head(10))

print(f"\nOutput: {output_path}")
print("Original Cohere GT was not modified.")

Cleaned Cohere GT saved successfully.

===== COHERE CLEANING SUMMARY =====
Original entities in valid articles: 6,799
Removed reference entities: 72
Cleaned entities: 6,727
Entities removed: 1.06%
Evaluation paragraphs excluded: 138
Pages with References cutoff: 158
Pages with External links cutoff: 1

===== REMOVED ENTITIES BY CUTOFF =====
references: 72 entities

===== REMOVED ENTITY PREVIEW =====


,Source,Index,word,entity_group
470,Murder of Pentecostals in Sloviansk,28,"Karpitsky, Nikolai Nikolayevich",PER
471,Murder of Pentecostals in Sloviansk,28,Donbas,LOC
472,Murder of Pentecostals in Sloviansk,28,Eastern Ukraine,LOC
473,Murder of Pentecostals in Sloviansk,28,Mission Eurasia,ORG
474,Murder of Pentecostals in Sloviansk,29,Богдан Красовский,PER
475,Murder of Pentecostals in Sloviansk,29,Славянске,LOC
476,Murder of Pentecostals in Sloviansk,29,Славянск,LOC
477,Murder of Pentecostals in Sloviansk,30,Дмитрий Фионик,PER
478,Murder of Pentecostals in Sloviansk,30,Славянске,LOC
479,Murder of Pentecostals in Sloviansk,30,ООО «Фокус Медиа»,ORG



===== CLEANED GT PREVIEW =====


,entity_group,score,word,start,end,Source,Index,Text
0,ORG,NaN,Ternopil Battalion,4,22,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
1,ORG,NaN,Special Tasks Patrol Police,51,78,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
2,ORG,NaN,Ministry of Internal Affairs of Ukraine,95,134,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
3,LOC,NaN,Ternopil,162,170,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
4,MISC,NaN,War in Donbass,218,232,Ternopil Rifle Battalion (Ukraine),0,The Ternopil Battalion is a rifle battalion of...
5,LOC,NaN,Ternopil Oblast,100,115,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
6,MISC,NaN,Euromaidan,216,226,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
7,PER,NaN,Vyacheslav Kizilov,296,314,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
8,MISC,NaN,ATO,420,423,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."
9,MISC,NaN,ATO,491,494,Ternopil Rifle Battalion (Ukraine),1,"It was established on 24 April 2014, in order ..."



Output: C:\Users\idosc\Corpus2GeoRAG\ApplyNER\NER_output\model_evaluation\ground_truth\cohere_entities_cleaned.parquet
Original Cohere GT was not modified.


Evaluation

In [10]:
# ============================================================
# 1. CONFIGURATION AND FILE PATHS
# ============================================================

LABELS = ("PER", "ORG", "LOC", "MISC")

evaluation_dir = (
    Path(REPO_PATH)
    / "ApplyNER"
    / "NER_output"
    / "model_evaluation"
)

gt_dir = evaluation_dir / "ground_truth"
model_dir = evaluation_dir / "NER_models_evaluation"

original_gt_path = gt_dir / "cohere_entities.parquet"
cleaned_gt_path = gt_dir / "cohere_entities_cleaned.parquet"
subset_path = evaluation_dir / "evaluation_subset.parquet"

flair_path = model_dir / "flair_entities.parquet"

cohere_raw_path = gt_dir / "cohere_command_a_03_2025_raw.jsonl"
flair_raw_path = model_dir / "flair_raw.jsonl"

required_files = {
    "Original Cohere GT": original_gt_path,
    "Cleaned Cohere GT": cleaned_gt_path,
    "Evaluation subset": subset_path,
    "Flair predictions": flair_path,
    "Cohere checkpoint": cohere_raw_path,
    "Flair checkpoint": flair_raw_path,
}

for name, path in required_files.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} not found: {path}")

# Confirm that preparation and cleaning were completed.
if "valid_pages" not in globals():
    raise RuntimeError(
        "Run the Uploading and Preparing GT cell first."
    )

if "excluded_paragraphs" not in globals():
    raise RuntimeError(
        "Run the Cleaning cell first."
    )

valid_page_set = set(valid_pages)

if not valid_page_set:
    raise ValueError("No valid Wikipedia articles available.")

excluded_keys = {
    (str(source), int(index))
    for source, index in excluded_paragraphs
}

# ============================================================
# 2. LOAD DATASETS
# ============================================================

gt_original = pd.read_parquet(original_gt_path)
gt_cleaned = pd.read_parquet(cleaned_gt_path)
flair_predictions = pd.read_parquet(flair_path)
subset = pd.read_parquet(subset_path)

entity_columns = {
    "Source", "Index", "Text",
    "word", "entity_group", "start", "end"
}

for name, df in [
    ("Original Cohere GT", gt_original),
    ("Cleaned Cohere GT", gt_cleaned),
    ("Flair predictions", flair_predictions),
]:
    missing = entity_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )

if not {"Source", "Index", "Text"}.issubset(subset.columns):
    raise ValueError("Evaluation subset has missing columns.")

if subset.duplicated(["Source", "Index"]).any():
    raise ValueError(
        "Duplicate paragraph identifiers in evaluation subset."
    )

# ============================================================
# 3. LOAD COMPLETED PARAGRAPH CHECKPOINTS
# ============================================================

def load_latest_records(path):
    """
    Load existing model checkpoints.

    If a paragraph has multiple saved records,
    use the latest record.
    """

    latest = {}

    with path.open("r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, 1):

            if not line.strip():
                continue

            try:
                record = json.loads(line)

                key = (
                    str(record["Source"]),
                    int(record["Index"]),
                    str(record["Text"])
                )

            except (ValueError, KeyError, TypeError) as exc:
                raise ValueError(
                    f"Invalid checkpoint in {path.name}, "
                    f"line {line_number}"
                ) from exc

            latest[key] = record

    return latest


cohere_records = load_latest_records(cohere_raw_path)
flair_records = load_latest_records(flair_raw_path)


def cohere_is_complete(record):
    """
    Verify that a saved Cohere response completed
    successfully and contains a valid entities array.
    """

    if record.get("model") != "command-a-03-2025":
        return False

    finish_reason = str(
        record.get("finish_reason", "")
    ).upper()

    if "COMPLETE" not in finish_reason:
        return False

    raw_text = record.get("raw_response")

    if not isinstance(raw_text, str) or not raw_text:
        raw_text = "".join(
            block.get("text", "")
            for block in (record.get("response_content") or [])
            if (
                isinstance(block, dict)
                and block.get("type") == "text"
                and isinstance(block.get("text"), str)
            )
        ).strip()

    try:
        parsed = json.loads(raw_text)
    except (TypeError, ValueError):
        return False

    return (
        isinstance(parsed, dict)
        and isinstance(parsed.get("entities"), list)
    )


cohere_completed = {
    key
    for key, record in cohere_records.items()
    if cohere_is_complete(record)
}

flair_completed = {
    key
    for key, record in flair_records.items()
    if (
        isinstance(record.get("settings"), dict)
        and record["settings"].get("model") == "flair/ner-english"
        and isinstance(record.get("entities"), list)
    )
}

# ============================================================
# 4. DEFINE THE VALID EVALUATION POPULATION
# ============================================================

evaluation_keys = {
    (str(r.Source), int(r.Index), str(r.Text))
    for r in subset.itertuples(index=False)
}

# Only paragraphs processed by both models.
common_paragraphs = (
    evaluation_keys
    & cohere_completed
    & flair_completed
)

if not common_paragraphs:
    raise ValueError(
        "No shared completed evaluation paragraphs found."
    )

# BEFORE CLEANING:
# Include only pages verified against the updated corpus.
# Reference paragraphs are still included.
original_scope = {
    key
    for key in common_paragraphs
    if key[0] in valid_page_set
}

# AFTER CLEANING:
# Use the same valid articles, excluding paragraphs
# removed by References / External links filtering.
cleaned_scope = {
    key
    for key in original_scope
    if (key[0], key[1]) not in excluded_keys
}

if not original_scope:
    raise ValueError(
        "No valid evaluation paragraphs remain."
    )

if not cleaned_scope:
    raise ValueError(
        "No paragraphs remain after reference cleaning."
    )

# ============================================================
# 5. VALIDATE DATASET CONSISTENCY
# ============================================================

evaluation_text = {
    (str(r.Source), int(r.Index)): str(r.Text)
    for r in subset.itertuples(index=False)
}


def get_entity_rows(df, scope, name):
    """
    Retrieve exact-match entity annotations
    belonging to the selected evaluation paragraphs.
    """

    entities = set()

    for row in df.itertuples(index=False):

        source = str(row.Source)
        index = int(row.Index)
        text = str(row.Text)

        pair = (source, index)
        paragraph = (source, index, text)

        if (
            pair in evaluation_text
            and evaluation_text[pair] != text
        ):
            raise ValueError(
                f"{name}: paragraph text mismatch "
                f"for {source}, index {index}"
            )

        if paragraph not in scope:
            continue

        start = int(row.start)
        end = int(row.end)
        label = str(row.entity_group)
        word = str(row.word)

        if label not in LABELS:
            raise ValueError(
                f"{name}: unexpected entity label {label}"
            )

        if not (0 <= start < end <= len(text)):
            raise ValueError(
                f"{name}: invalid entity offsets "
                f"for {source}, index {index}"
            )

        if text[start:end] != word:
            raise ValueError(
                f"{name}: entity text/offset mismatch "
                f"for {source}, index {index}"
            )

        entity = (
            source, index, text,
            start, end, label
        )

        if entity in entities:
            raise ValueError(
                f"{name}: duplicate entity annotation"
            )

        entities.add(entity)

    return entities


# ============================================================
# 6. VERIFY THE CLEANED GT
# ============================================================

def annotation_counts(df):
    """
    Count annotations independently of row order.
    Preserve duplicate counts during comparison.
    """

    return Counter(
        (
            str(r.Source),
            int(r.Index),
            str(r.Text),
            int(r.start),
            int(r.end),
            str(r.entity_group),
            str(r.word)
        )
        for r in df.itertuples(index=False)
    )


# The cleaned GT should contain ONLY:
# 1. Entities from verified articles.
# 2. Entities from paragraphs before the cutoff.
#
# Nothing else should have changed.

expected_cleaned = gt_original.loc[
    gt_original["Source"].isin(valid_page_set)
    & pd.Series(
        [
            (str(source), int(index)) not in excluded_keys
            for source, index in zip(
                gt_original["Source"],
                gt_original["Index"]
            )
        ],
        index=gt_original.index
    )
]

if annotation_counts(expected_cleaned) != annotation_counts(gt_cleaned):
    raise ValueError(
        "Cleaned Cohere GT contains unexpected changes. "
        "It must differ from the original only through "
        "excluded articles and reference paragraphs."
    )

print("Cleaned Cohere GT validation passed.")

# ============================================================
# 7. CALCULATE EXACT-MATCH METRICS
# ============================================================

def calculate_metrics(gt_entities, predicted_entities):

    true_positives = gt_entities & predicted_entities
    false_positives = predicted_entities - gt_entities
    false_negatives = gt_entities - predicted_entities

    def scores(tp, fp, fn):

        precision = (
            tp / (tp + fp)
            if tp + fp else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn else 0.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        return (
            precision * 100,
            recall * 100,
            f1 * 100
        )

    tp = len(true_positives)
    fp = len(false_positives)
    fn = len(false_negatives)

    precision, recall, f1 = scores(tp, fp, fn)

    per_label = []

    for label in LABELS:

        label_tp = sum(
            entity[-1] == label
            for entity in true_positives
        )

        label_fp = sum(
            entity[-1] == label
            for entity in false_positives
        )

        label_fn = sum(
            entity[-1] == label
            for entity in false_negatives
        )

        p, r, label_f1 = scores(
            label_tp, label_fp, label_fn
        )

        per_label.append({
            "Label": label,
            "Support": label_tp + label_fn,
            "Precision (%)": p,
            "Recall (%)": r,
            "F1 (%)": label_f1
        })

    macro_f1 = sum(
        row["F1 (%)"]
        for row in per_label
    ) / len(LABELS)

    return {
        "GT entities": len(gt_entities),
        "Predicted entities": len(predicted_entities),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision (%)": precision,
        "Recall (%)": recall,
        "Micro F1 (%)": f1,
        "Macro F1 (%)": macro_f1,
    }, per_label


# ============================================================
# 8. EVALUATE BEFORE CLEANING
# ============================================================

original_truth = get_entity_rows(
    gt_original,
    original_scope,
    "Original Cohere GT"
)

original_predictions = get_entity_rows(
    flair_predictions,
    original_scope,
    "Flair"
)

original_metrics, original_per_label = calculate_metrics(
    original_truth,
    original_predictions
)

# ============================================================
# 9. EVALUATE AFTER CLEANING
# ============================================================

cleaned_truth = get_entity_rows(
    gt_cleaned,
    cleaned_scope,
    "Cleaned Cohere GT"
)

cleaned_predictions = get_entity_rows(
    flair_predictions,
    cleaned_scope,
    "Flair"
)

cleaned_metrics, cleaned_per_label = calculate_metrics(
    cleaned_truth,
    cleaned_predictions
)

# ============================================================
# 10. DISPLAY EVALUATION COVERAGE
# ============================================================

print("\n===== EVALUATION COVERAGE =====")

print(f"Original evaluation paragraphs: {len(evaluation_keys):,}")

print(f"Cohere completed paragraphs: {len(cohere_completed):,}")
print(f"Flair completed paragraphs: {len(flair_completed):,}")

print(f"Shared completed paragraphs: {len(common_paragraphs):,}")

print(
    "Excluded due to corpus changes: "
    f"{len(common_paragraphs - original_scope):,}"
)

print(
    "Valid evaluation paragraphs before cleaning: "
    f"{len(original_scope):,}"
)

print(
    "Reference paragraphs excluded: "
    f"{len(original_scope - cleaned_scope):,}"
)

print(
    "Evaluation paragraphs after cleaning: "
    f"{len(cleaned_scope):,}"
)

print(f"Valid Wikipedia articles: {len(valid_page_set):,}")

# ============================================================
# 11. OVERALL EVALUATION
# ============================================================

print("\n===== OVERALL EVALUATION =====")

comparison = pd.DataFrame(
    [
        {
            "Ground Truth": "Original Cohere",
            "Paragraphs": len(original_scope),
            **original_metrics
        },
        {
            "Ground Truth": "Cleaned Cohere",
            "Paragraphs": len(cleaned_scope),
            **cleaned_metrics
        }
    ]
).set_index("Ground Truth")

display(comparison.round(2))

# ============================================================
# 12. PERFORMANCE CHANGE
# ============================================================

print("\n===== PERFORMANCE CHANGE =====")

metric_columns = [
    "Precision (%)",
    "Recall (%)",
    "Micro F1 (%)",
    "Macro F1 (%)"
]

changes = pd.DataFrame({
    "Metric": metric_columns,
    "Before (%)": [
        original_metrics[column]
        for column in metric_columns
    ],
    "After (%)": [
        cleaned_metrics[column]
        for column in metric_columns
    ]
})

changes["Difference (pp)"] = (
    changes["After (%)"] - changes["Before (%)"]
)

display(changes.round(2))

# ============================================================
# 13. PER-ENTITY-TYPE EVALUATION
# ============================================================

print("\n===== PER-ENTITY-TYPE EVALUATION =====")

per_label_comparison = pd.concat(
    [
        pd.DataFrame(original_per_label).assign(
            Ground_Truth="Original Cohere"
        ),
        pd.DataFrame(cleaned_per_label).assign(
            Ground_Truth="Cleaned Cohere"
        )
    ],
    ignore_index=True
)

display(
    per_label_comparison[
        [
            "Ground_Truth",
            "Label",
            "Support",
            "Precision (%)",
            "Recall (%)",
            "F1 (%)"
        ]
    ].round(2)
)

print("\nEvaluation completed.")
print("No model inference or API calls.")
print("Original files remain unchanged.")

Cleaned Cohere GT validation passed.

===== EVALUATION COVERAGE =====
Original evaluation paragraphs: 6,565
Cohere completed paragraphs: 979
Flair completed paragraphs: 979
Shared completed paragraphs: 979
Excluded due to corpus changes: 555
Valid evaluation paragraphs before cleaning: 424
Reference paragraphs excluded: 1
Evaluation paragraphs after cleaning: 423
Valid Wikipedia articles: 123

===== OVERALL EVALUATION =====


,Paragraphs,GT entities,Predicted entities,TP,FP,FN,Precision (%),Recall (%),Micro F1 (%),Macro F1 (%)
Ground Truth,,,,,,,,,,
Original Cohere,424,3579,3923,3016,907,563,76.88,84.27,80.41,76.17
Cleaned Cohere,423,3575,3923,3016,907,559,76.88,84.36,80.45,76.23



===== PERFORMANCE CHANGE =====


,Metric,Before (%),After (%),Difference (pp)
0,Precision (%),76.88,76.88,0.00
1,Recall (%),84.27,84.36,0.09
2,Micro F1 (%),80.41,80.45,0.04
3,Macro F1 (%),76.17,76.23,0.06



===== PER-ENTITY-TYPE EVALUATION =====


,Ground_Truth,Label,Support,Precision (%),Recall (%),F1 (%)
0,Original Cohere,PER,202,63.89,91.09,75.10
1,Original Cohere,ORG,624,71.41,72.44,71.92
2,Original Cohere,LOC,1963,92.20,88.54,90.33
3,Original Cohere,MISC,790,57.48,81.27,67.33
4,Cleaned Cohere,PER,202,63.89,91.09,75.10
5,Cleaned Cohere,ORG,620,71.41,72.90,72.15
6,Cleaned Cohere,LOC,1963,92.20,88.54,90.33
7,Cleaned Cohere,MISC,790,57.48,81.27,67.33



Evaluation completed.
No model inference or API calls.
Original files remain unchanged.


### Commit to Repo

In [11]:
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

LOCAL_REPO_PATH = Path.home() / GITHUB_REPO
MAX_GIT_FILE_BYTES = 100 * 1024 * 1024

# --------------------------------------------------
# Detect runtime
# --------------------------------------------------

if os.name == "nt":
    IS_LOCAL = True
elif os.environ.get("COLAB_RELEASE_TAG"):
    IS_LOCAL = False
else:
    raise RuntimeError(
        "Could not identify this runtime as Windows or a hosted Colab VM."
    )

if shutil.which("git") is None:
    raise RuntimeError(
        "Git is not available to this Jupyter environment. "
        "Install Git for Windows and restart Jupyter if running locally."
    )

# --------------------------------------------------
# Locate the repository
# --------------------------------------------------

if IS_LOCAL:
    candidates = [Path.cwd(), LOCAL_REPO_PATH]

    previous_repo_path = globals().get("REPO_PATH")
    if previous_repo_path:
        candidates.insert(0, Path(previous_repo_path))

    detected_repo = None

    for candidate in candidates:
        if not candidate.is_dir():
            continue

        probe = subprocess.run(
            ["git", "rev-parse", "--show-toplevel"],
            cwd=candidate,
            text=True,
            encoding="utf-8",
            errors="replace",
            capture_output=True
        )

        if probe.returncode == 0:
            root = Path(probe.stdout.strip()).resolve()

            if root.name.lower() == GITHUB_REPO.lower():
                detected_repo = root
                break

    if detected_repo is None:
        raise FileNotFoundError(
            f"Could not find your local {GITHUB_REPO} checkout.\n"
            f"Clone it to {LOCAL_REPO_PATH}, or update LOCAL_REPO_PATH."
        )

    REPO_PATH = str(detected_repo)

else:
    REPO_PATH = f"/content/{GITHUB_REPO}"

    if not Path(REPO_PATH).is_dir():
        raise FileNotFoundError(
            f"{REPO_PATH} does not exist. Run your clone cell first."
        )

os.chdir(REPO_PATH)

print("Runtime:", "Local Windows" if IS_LOCAL else "Colab VM")
print("Repository:", REPO_PATH)


def run_git(args, *, env=None, show_output=True):
    """Run Git and include its actual output if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_PATH,
        env=env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if show_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        details = (result.stderr or result.stdout or "").strip()
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    show_output=False
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

print("Branch:", branch)

# Stop if an earlier rebase or merge is still paused.
for state_path in ("rebase-merge", "rebase-apply", "MERGE_HEAD"):
    git_state = run_git(
        ["rev-parse", "--git-path", state_path],
        show_output=False
    ).stdout.strip()

    if Path(git_state).exists():
        raise RuntimeError(
            "An earlier rebase or merge is still in progress. "
            "Check `git status` and resolve it before rerunning this cell."
        )

# --------------------------------------------------
# Authentication and synchronization
# --------------------------------------------------

git_env = os.environ.copy()
git_env["GIT_TERMINAL_PROMPT"] = "0"

auth_options = []
askpass_dir = None

try:
    if IS_LOCAL:
        # Use Git credentials already configured on your computer.
        print("Authentication: local Git credentials.")

    else:
        from google.colab import userdata

        github_token = userdata.get("GITHUB_TOKEN")

        if not github_token:
            raise ValueError(
                "GITHUB_TOKEN is missing from Colab Secrets. "
                "Add it and enable notebook access."
            )

        askpass_dir = tempfile.TemporaryDirectory(
            prefix="github-askpass-"
        )
        askpass_path = Path(askpass_dir.name) / "askpass.sh"

        askpass_path.write_text(
            "#!/bin/sh\n"
            'case "$1" in\n'
            '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
            '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
            "esac\n",
            encoding="utf-8"
        )
        askpass_path.chmod(0o700)

        git_env["GITHUB_USER"] = GITHUB_USER
        git_env["GITHUB_TOKEN"] = github_token
        git_env["GIT_ASKPASS"] = str(askpass_path)

        auth_options = ["-c", "credential.helper="]

        print("Authentication: Colab GITHUB_TOKEN Secret.")

    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    ignore_entry = f"/{GITHUB_REPO}/"
    gitignore_path = Path(REPO_PATH) / ".gitignore"

    existing_ignore = (
        gitignore_path.read_text(encoding="utf-8")
        if gitignore_path.exists()
        else ""
    )

    if ignore_entry not in existing_ignore.splitlines():
        with gitignore_path.open("a", encoding="utf-8") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    nested_repo = run_git(
        ["ls-files", "--stage", "--", GITHUB_REPO],
        show_output=False
    ).stdout.strip()

    if nested_repo:
        run_git([
            "rm", "--cached", "-r", "-f", "--", GITHUB_REPO
        ])
        print("Removed the nested repository path from Git tracking.")

    # --------------------------------------------------
    # Find files exceeding GitHub's per-file limit
    # --------------------------------------------------

    candidate_files = run_git(
        [
            "ls-files",
            "--cached",
            "--others",
            "--exclude-standard",
            "-z"
        ],
        show_output=False
    ).stdout.split("\0")

    excluded_files = set()

    for relative_path in sorted(set(candidate_files) - {""}):
        file_path = Path(REPO_PATH) / relative_path

        if not file_path.is_file():
            continue

        file_size = file_path.stat().st_size

        if file_size > MAX_GIT_FILE_BYTES:
            excluded_files.add(relative_path)

            # Remove from Git's index, while keeping the file on disk.
            run_git(
                [
                    "rm",
                    "--cached",
                    "--ignore-unmatch",
                    "-f",
                    "--",
                    f":(top,literal){relative_path}"
                ],
                show_output=False
            )

            print(
                f"Excluded oversized file: {relative_path} "
                f"({file_size / 1024**2:.2f} MiB). "
                "Kept on disk."
            )

    # --------------------------------------------------
    # Stage changes, excluding oversized files
    # --------------------------------------------------

    exclusion_pathspecs = [
        f":(top,literal,exclude){path}"
        for path in sorted(excluded_files)
    ]

    run_git(["add", "-A", "--", ".", *exclusion_pathspecs])

    # Exclude notebook copies, preserving the original cell's behavior.
    staged_files = run_git(
        ["diff", "--cached", "--name-only", "-z"],
        show_output=False
    ).stdout.split("\0")

    for notebook in staged_files:
        if (
            notebook.endswith(".ipynb")
            and notebook not in excluded_files
        ):
            run_git([
                "restore", "--staged", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Create a normal commit
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        cwd=REPO_PATH,
        stdin=subprocess.DEVNULL
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebook copies on disk
    # This discards edits to those on-disk copies.
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "-z", "HEAD"],
        show_output=False
    ).stdout.split("\0")

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git([
                "restore", "--worktree", "--",
                f":(top,literal){notebook}"
            ])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", *auth_options, "pull", "--rebase", "origin", branch],
        cwd=REPO_PATH,
        env=git_env,
        stdin=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            show_output=False
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                show_output=False
            ).stdout

            theirs = run_git(
                ["show", ":3:.gitignore"],
                show_output=False
            ).stdout

            if ours.strip() != theirs.strip():
                raise RuntimeError(
                    "The .gitignore versions differ. The rebase is paused; "
                    "inspect the conflict before continuing."
                )

            gitignore_path.write_text(
                ours + ("\n" if ours and not ours.endswith("\n") else ""),
                encoding="utf-8"
            )

            run_git(["add", "--", ".gitignore"])
            run_git([
                "-c", "core.editor=true", "rebase", "--continue"
            ])

        else:
            raise RuntimeError(
                "Pull failed. Check the Git output above and `git status`. "
                "If running locally, verify your GitHub authentication."
            )

    # --------------------------------------------------
    # Push synchronized changes
    # --------------------------------------------------

    run_git(
        [*auth_options, "push", "origin", branch],
        env=git_env
    )

    print(
        f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} "
        f"on branch '{branch}'."
    )

finally:
    if askpass_dir is not None:
        askpass_dir.cleanup()

    git_env.pop("GITHUB_TOKEN", None)

Runtime: Local Windows
Repository: C:\Users\idosc\Corpus2GeoRAG
Branch: main
Authentication: local Git credentials.
[main 7de6d67] Update NER workflow
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 ApplyNER/NER_output/model_evaluation/ground_truth/cohere_entities_cleaned.parquet
Repository changes committed.
Current branch main is up to date.
From https://github.com/idoschw3/Corpus2GeoRAG
 * branch            main       -> FETCH_HEAD
To https://github.com/idoschw3/Corpus2GeoRAG.git
   4a6f214..7de6d67  main -> main
Successfully pushed to idoschw3/Corpus2GeoRAG on branch 'main'.
